In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

df = pd.read_csv("hong_kong_salary.csv") 
df.columns = [c.strip() for c in df.columns]

LEVEL_COL  = "Level of Study"        
PROG_COL   = "Broad Academic Programme Category"    
SALARY_COL = "Average Annual Salary (HK$'000)"  


if pd.to_numeric(df[SALARY_COL], errors="coerce").median() < 10000:
    df[SALARY_COL] = pd.to_numeric(df[SALARY_COL], errors="coerce") * 1000
else:
    df[SALARY_COL] = pd.to_numeric(df[SALARY_COL], errors="coerce")

df = df.dropna(subset=[SALARY_COL])


def summary_table(frame, group_col, val_col):
    g = frame.groupby(group_col, dropna=False)[val_col]
    q25 = g.quantile(0.25)
    q75 = g.quantile(0.75)
    out = pd.DataFrame({
        "n": g.size(),
        "mean": g.mean(),
        "median": g.median(),
        "sd": g.std(),
        "IQR": q75 - q25,
        "min": g.min(),
        "max": g.max()
    }).reset_index()

    return out.sort_values(group_col).reset_index(drop=True)

table_level = summary_table(df, LEVEL_COL, SALARY_COL)
table_prog  = summary_table(df, PROG_COL,  SALARY_COL)

print("\n=== Summary by Level of Study ===")
print(table_level.to_string(index=False))
print("\n=== Summary by Programme/Discipline ===")
print(table_prog.to_string(index=False))

level_map = {
    "Sub-degree": 1, "Associate Degree": 1, "Diploma": 1, "Higher Diploma": 1,
    "Undergraduate": 2, "Bachelor": 2, "Bachelor's": 2, "UG": 2,
    "Taught Postgraduate": 3, "TPG": 3, "Master": 3, "Master's": 3,
    "Research Postgraduate": 4, "RPG": 4, "Doctoral": 4, "PhD": 4
}

df["Level_code"] = df[LEVEL_COL].map(level_map)
unmapped = df.loc[df["Level_code"].isna(), LEVEL_COL].dropna().unique().tolist()
if unmapped:
    print("\n[Note] Unmapped level labels found. Update 'level_map' for:", unmapped)


rank_df = df.dropna(subset=["Level_code", SALARY_COL]).copy()

rho, pval = spearmanr(rank_df["Level_code"], rank_df[SALARY_COL])


def spearman_bootstrap_ci(x, y, n_boot=5000, ci=0.95, random_state=42):
    rng = np.random.default_rng(random_state)
    xy = np.column_stack([x, y])
    xy = xy[~np.isnan(xy).any(axis=1)]
    n = xy.shape[0]
    boots = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        xb, yb = xy[idx, 0], xy[idx, 1]
        boots[i] = spearmanr(xb, yb).correlation
    alpha = (1 - ci) * 100 / 2
    lo, hi = np.percentile(boots, [alpha, 100 - alpha])
    return lo, hi

ci_lo, ci_hi = spearman_bootstrap_ci(rank_df["Level_code"].values,
                                     rank_df[SALARY_COL].values,
                                     n_boot=5000, ci=0.95, random_state=1)

print(f"\nSpearman rho (Level vs Salary) = {rho:.3f}, 95% CI [{ci_lo:.3f}, {ci_hi:.3f}], p = {pval:.3g}")


caption = (f"ρ = {rho:.3f}, 95% CI [{ci_lo:.3f}, {ci_hi:.3f}], p = {pval:.3g}")
print("\nFigure caption add-on ->", caption)


=== Summary by Level of Study ===
       Level of Study   n          mean   median           sd      IQR    min    max
Research Postgraduate 105 339561.904762 340000.0 72662.672119 109000.0 227000 524000
           Sub-degree  91 193032.967033 182000.0 52259.066316  69500.0 120000 388000
  Taught Postgraduate  92 365119.565217 351000.0 99662.600301 137750.0 219000 725000
        Undergraduate 105 257952.380952 234000.0 89905.225842  98000.0 148000 540000

=== Summary by Programme/Discipline ===
Broad Academic Programme Category  n          mean   median            sd      IQR    min    max
              Arts and Humanities 60 256150.000000 242500.0  85676.534968 109500.0 120000 456000
          Business and Management 43 282209.302326 246000.0 126543.905074 190000.0 129000 524000
                        Education 60 291550.000000 281000.0  77156.988325 119250.0 155000 435000
       Engineering and Technology 60 269283.333333 262000.0  83309.732816 141750.0 125000 448000
   Medicine, D

In [6]:
from scipy.stats import spearmanr
import pandas as pd

df = pd.read_csv('hong_kong_salary.csv')

order = ["Sub-degree","Undergraduate","Taught Postgraduate","Research Postgraduate"]
code = pd.Categorical(df["Level of Study"], categories=order, ordered=True).codes + 1

rho, p = spearmanr(code, df["Average Annual Salary (HK$'000)"], alternative="greater")
print("Spearman rho =", rho, "p =", p)


Spearman rho = 0.6416870793150458 p = 2.774840592503092e-47
